# Generative Model Interpretability and Text Simplification Framework

Master's Thesis Research  
Author: Davinson Poveda  
Core Stack: Python, Transformers, Advanced XAI (SHAP, LIME), SyntaxSHAP, SpaCy, EASSE

[→ Open Interactive Notebook in Google Colab](https://colab.research.google.com/github/davinsonpoveda/Generative-Model-Interpretability-and-Text-Simplification-Framework/blob/main/experimentos_XAI_2026.ipynb)

---

1. Configuración del Entorno y Dependencias
Este bloque instala las librerías necesarias para el manejo de Transformers, los explicadores XAI y el procesamiento sintáctico del español.

In [ ]:
import os
os._exit(0)

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets captum shap lime
# Nota: SyntaxSHAP suele requerir su propia implementación o manejo de máscaras sintácticas
!pip install -q -U bitsandbytes>=0.46.1 transformers accelerate peft
import torch
import pandas as pd
import spacy
import shap
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoModelForSeq2SeqLM, BitsAndBytesConfig
from transformers import BitsAndBytesConfig
from huggingface_hub import login
from lime.lime_text import LimeTextExplainer

# 1. AUTENTICACIÓN Y VARIABLES GLOBALES

hf_token = "tu token aqui"
login(token=hf_token)

# Cargar spaCy para SyntaxSHAP y métricas sintácticas
try:
    nlp = spacy.load("es_core_news_sm")
except:
    !python -m spacy download es_core_news_sm
    nlp = spacy.load("es_core_news_sm")

device = "cuda" if torch.cuda.is_available() else "cpu"

2. Carga de Modelos y Dataset Clara-MeD
Cargaremos BART-large (representando el paradigma Encoder-Decoder
) y LSLlama (representando el paradigma Decoder-only
).

In [ ]:
from google.colab import drive
import os

# Montamos tu Drive
drive.mount('/content/drive')

# Definimos la ruta de tu carpeta basándonos en la imagen
path_tfm = "/content/drive/MyDrive/UC3m_TFM/"

# Creamos subcarpetas para organizar tu TFM
path_modelos = os.path.join(path_tfm, "modelos")
path_resultados = os.path.join(path_tfm, "resultados")

os.makedirs(path_modelos, exist_ok=True)
os.makedirs(path_resultados, exist_ok=True)
# Necesitamos definir esto para que la carga local funcione
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)


print(f"✅ Rutas configuradas. Los modelos se guardarán en: {path_modelos}")

3. Cargar el modelo en segundos (El resto de los días)
A partir de que esté guardado, tu código de carga será este (mucho más rápido y seguro):

In [ ]:
print("--- Cargando Arquitecturas desde Drive ---")

# A. LSLlama desde DRIVE (No descarga de internet)
ruta_llama_drive = os.path.join(path_modelos, "LSLlama_4bit")
llama_model = AutoModelForCausalLM.from_pretrained(
    ruta_llama_drive,
    quantization_config=bnb_config,
    device_map="auto"
)
llama_tokenizer = AutoTokenizer.from_pretrained(ruta_llama_drive)

# B. BART (Carga rápida)
bart_id = "facebook/bart-large-cnn"
bart_tokenizer = AutoTokenizer.from_pretrained(bart_id)
bart_model = AutoModelForSeq2SeqLM.from_pretrained(bart_id).to("cuda")

print("✅ Modelos listos para la comparativa.")

con el dataset de claramed

In [ ]:
import os
print("Archivos detectados en tu Drive:")
print(os.listdir("/content/drive/MyDrive/UC3m_TFM/"))

In [ ]:
import pandas as pd
import os
import numpy as np
import random

#Configuración de Reproducibilidad
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Esto asegura que los algoritmos de la GPU sean deterministas
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) # Usamos 42 como estándar

# 1. Cargar el dataset
path_tsv = os.path.join(path_tfm, "claramed_synt_simp_aligned.tsv")
df_claramed = pd.read_csv(path_tsv, sep='\t')

# 2. DETECCIÓN AUTOMÁTICA DE COLUMNAS
# Usamos .columns[0] para el original y .columns[1] para el humano
col_original = df_claramed.columns[1]
col_referencia = df_claramed.columns[3]

print(f"📊 Columnas detectadas:")
print(f"   - Original: '{col_original}'")
print(f"   - Referencia: '{col_referencia}'")

# Tomamos una muestra de 5 frases
muestra = df_claramed.head(5)

print(f"\n🚀 Procesando {len(muestra)} frases de ClaraMeD...\n")

resultados_tfm = []

for index, row in muestra.iterrows():
    frase_original = str(row[col_original])
    referencia_humana = str(row[col_referencia])

    # --- Inferencia LSLlama ---
    prompt = f"### Instrucción: Simplifica el siguiente texto médico al lenguaje común.\n\n### Texto: {frase_original}\n\n### Simplificación:"
    inputs_llama = llama_tokenizer(prompt, return_tensors="pt").to("cuda")

    #justificar el modelo con respecto al articulo de Shap
    with torch.no_grad():
        out_llama = llama_model.generate(
            **inputs_llama,
            max_new_tokens=80,
            temperature=0.1,
            repetition_penalty=1.2,
            do_sample=True
        )
    res_llama = llama_tokenizer.decode(out_llama[0], skip_special_tokens=True).split("### Simplificación:")[-1].strip()

    # --- Inferencia BART ---
    inputs_bart = bart_tokenizer(frase_original, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out_bart = bart_model.generate(**inputs_bart, max_length=80, min_length=5, num_beams=4)
    res_bart = bart_tokenizer.decode(out_bart[0], skip_special_tokens=True)

    resultados_tfm.append({
        "Original (ClaraMeD)": frase_original,
        "LSLlama (IA)": res_llama,
        "BART (IA)": res_bart,
        "Humano (Gold Standard)": referencia_humana
    })

# 3. Crear DataFrame y mostrar
df_final = pd.DataFrame(resultados_tfm)
pd.set_option('display.max_colwidth', None)
display(df_final)

# Guardar en Drive
df_final.to_csv(os.path.join(path_resultados, "test_claramed_final.csv"), index=False, sep=';')
print("\n✅ ¡Misión cumplida! Tabla generada y guardada.")

Realizando la simplificacion con Bart y Lsllama para todo el dataset de CLARAMED

In [ ]:
import pandas as pd
import os
import numpy as np
import random
import torch
from tqdm import tqdm

# Configuración de Reproducibilidad
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# ==========================================
# CONFIGURACIÓN ESTRICTA PARA BATCHING SEGURO
# ==========================================
if llama_tokenizer.pad_token is None:
    llama_tokenizer.pad_token = llama_tokenizer.eos_token
llama_tokenizer.padding_side = "left"  # Obligatorio para modelos Decoder-only como Llama

if bart_tokenizer.pad_token is None:
    bart_tokenizer.pad_token = bart_tokenizer.eos_token
bart_tokenizer.padding_side = "right" # Recomendado para modelos Encoder-Decoder como BART

# Asegurar que el modelo conoce el token de padding internamente
llama_model.config.pad_token_id = llama_tokenizer.pad_token_id
bart_model.config.pad_token_id = bart_tokenizer.pad_token_id

# 1. Cargar el dataset completo
path_tsv = os.path.join(path_tfm, "claramed_synt_simp_aligned.tsv")
df_claramed = pd.read_csv(path_tsv, sep='\t')

col_original = df_claramed.columns[1]
col_referencia = df_claramed.columns[3]

# Parámetro de paralelismo
BATCH_SIZE = 16
total_frases = len(df_claramed)

print(f"\n🚀 Procesando dataset completo por LOTES (Batch Size: {BATCH_SIZE})...")
print(f"⚠️ Manteniendo hiperparámetros originales: LSLlama (do_sample=True, temp=0.1) y BART (num_beams=4).\n")

resultados_tfm = []

# Bucle por lotes
for i in tqdm(range(0, total_frases, BATCH_SIZE), desc="Lotes completados"):

    # Forzamos la semilla exacta al inicio de cada lote
    set_seed(42)

    # Segmentamos el lote actual
    batch_rows = df_claramed.iloc[i:i+BATCH_SIZE]

    list_originales = batch_rows[col_original].astype(str).tolist()
    list_referencias = batch_rows[col_referencia].astype(str).tolist()

    # --- Inferencia LSLlama por Lote ---
    prompts_llama = [
        f"### Instrucción: Simplifica el siguiente texto médico al lenguaje común.\n\n### Texto: {frase}\n\n### Simplificación:"
        for frase in list_originales
    ]

    # SOLUCIÓN 1: Tokenización con padding Y TRUNCADO de seguridad (máximo 1024 tokens)
    inputs_llama = llama_tokenizer(
        prompts_llama,
        return_tensors="pt",
        padding=True,
        truncation=True,        # <--- Evita desbordamiento de posición en la GPU
        max_length=1024         # <--- Límite seguro para la ventana de contexto
    ).to("cuda")

    with torch.no_grad():
        out_llama = llama_model.generate(
            **inputs_llama,
            max_new_tokens=80,
            temperature=0.1,
            repetition_penalty=1.2,
            do_sample=True,
            pad_token_id=llama_tokenizer.pad_token_id,  # <--- SOLUCIÓN 2: Forzar ID de pad explícito
            eos_token_id=llama_tokenizer.eos_token_id
        )

    res_llama_batch = llama_tokenizer.batch_decode(out_llama, skip_special_tokens=True)
    res_llama_limpias = [texto.split("### Simplificación:")[-1].strip() for texto in res_llama_batch]

    # --- Inferencia BART por Lote ---
    # SOLUCIÓN 1b: Añadimos también truncado preventivo en BART
    inputs_bart = bart_tokenizer(
        list_originales,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=1024
    ).to("cuda")

    with torch.no_grad():
        out_bart = bart_model.generate(
            **inputs_bart,
            max_length=80,
            min_length=5,
            num_beams=4
        )

    res_bart_limpias = bart_tokenizer.batch_decode(out_bart, skip_special_tokens=True)

    # Reagrupar y guardar en la estructura final
    for idx in range(len(list_originales)):
        resultados_tfm.append({
            "Original (ClaraMeD)": list_originales[idx],
            "LSLlama (IA)": res_llama_limpias[idx],
            "BART (IA)": res_bart_limpias[idx],
            "Humano (Gold Standard)": list_referencias[idx]
        })

# 3. Crear DataFrame y guardar todo
df_final = pd.DataFrame(resultados_tfm)
path_salida_completo = os.path.join(path_resultados, "claramed_final_procesado.csv")
df_final.to_csv(path_salida_completo, index=False, sep=';', encoding='utf-8')

print(f"\n✅ ¡Procesamiento completado con éxito!")
print(f"💾 Archivo guardado de forma segura en: {path_salida_completo}")

Experimentos syntaxshap y shap

In [ ]:
import pandas as pd
import numpy as np
import shap
import spacy
import os
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer

# 1. CARGA DE DATOS (Mantenemos la lógica de índices que ya te funcionó)
path_real = 'drive/MyDrive/UC3m_TFM/resultados/test_claramed_final.csv'
df_test = pd.read_csv(path_real, sep=None, engine='python', on_bad_lines='skip')

frase_orig = str(df_test.iloc[0, 0])
resp_lsllama = str(df_test.iloc[0, 1])
resp_bart = str(df_test.iloc[0, 2])

# 2. CONFIGURACIÓN DE MODELOS Y TOKENIZER
nlp = spacy.load("es_core_news_sm")
model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# Usamos un tokenizador estándar que SHAP reconozca sin errores
tokenizer_shap = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

def get_score(texts, target):
    emb_target = model_sim.encode([target], convert_to_tensor=True)
    emb_inputs = model_sim.encode(texts, convert_to_tensor=True)
    scores = util.cos_sim(emb_inputs, emb_target).cpu().numpy().flatten()
    return scores

# --- EXPERIMENTO SHAP ---
print("\nIniciando SHAP (Atribución de importancia)...")

# El secreto está en pasar el tokenizer del modelo de similitud
masker = shap.maskers.Text(tokenizer_shap)
# importante ! explicar cada linea de codigo cada valor de parametro
# porque este modelo y no otra opcion.
explainer_l = shap.Explainer(lambda x: get_score(x, resp_lsllama), masker)
shap_l = explainer_l([frase_orig])

explainer_b = shap.Explainer(lambda x: get_score(x, resp_bart), masker)
shap_b = explainer_b([frase_orig])

# --- EXPERIMENTO SyntaxSHAP ---
print("Iniciando SyntaxSHAP...")
doc = nlp(frase_orig)
syntax_rows = []

# Mapeo de pesos. SHAP genera pesos por tokens del tokenizer_shap.
#  simplificar la visualización al número de palabras del doc original.
for i, token in enumerate(doc):
    # Tomamos el peso absoluto del token correspondiente
    p_l = abs(shap_l.values[0][i]) if i < len(shap_l.values[0]) else 0
    p_b = abs(shap_b.values[0][i]) if i < len(shap_b.values[0]) else 0
    syntax_rows.append({
        "Token": token.text,
        "POS": token.pos_,
        "Peso_LSLlama": p_l,
        "Peso_BART": p_b
    })

df_syntax = pd.DataFrame(syntax_rows)

# --- RESULTADOS FINALES ---
print("\n" + "="*45)
print("TABLA DE RESULTADOS SINTÁCTICOS (XAI)")
print("="*45)
resumen_pos = df_syntax.groupby("POS")[["Peso_LSLlama", "Peso_BART"]].mean()
display(resumen_pos.sort_values(by="Peso_LSLlama", ascending=False))

print("\nVisualización SHAP para LSLlama:")
shap.plots.text(shap_l[0])

Experimento LIME para LsLLama te dirá: "Si borro la palabra 'antagonistas', ¿qué tanto cambia la confianza de LSLlama?"

In [ ]:
import numpy as np
import pandas as pd
from lime.lime_text import LimeTextExplainer
from sentence_transformers import SentenceTransformer, util

# --- CONFIGURACIÓN PREVIA ---
# Usamos el modelo de similitud para medir qué tanto se aleja la frase perturbada
# de la simplificación que hizo LSLlama.
model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Extraemos los datos de la primera fila de tu df_final (el que generaste con el bucle for)
frase_orig = df_final.iloc[0]["Original (ClaraMeD)"]
resp_lsllama = df_final.iloc[0]["LSLlama (IA)"]

# --- CONFIGURACIÓN DE LIME ---
explainer_lime = LimeTextExplainer(class_names=['Diferente', 'Similar'])

# Función de predicción: LIME borrará palabras y evaluará contra la salida de LSLlama
def predict_fn_lsllama(texts):
    # Comparamos semánticamente las perturbaciones contra la salida REAL de LSLlama
    emb_target = model_sim.encode([resp_lsllama], convert_to_tensor=True)
    emb_inputs = model_sim.encode(texts, convert_to_tensor=True)

    # Calculamos similitud del coseno
    scores = util.cos_sim(emb_inputs, emb_target).cpu().numpy().flatten()

    # Formato LIME: probabilidades [Prob_Diferente, Prob_Similar]
    return np.array([[1-s, s] for s in scores])

# --- EJECUCIÓN DEL EXPERIMENTO ---
print(f"📊 Generando Auditoría LIME para LSLlama (Experimento 3)...")
print(f"🔍 Evaluando impacto sobre la simplificación: '{resp_lsllama[:80]}...'")

exp_lsllama = explainer_lime.explain_instance(
    frase_orig,
    predict_fn_lsllama,
    num_features=10,
    num_samples=500 # Vecindad de permutaciones
)

# --- VISUALIZACIÓN INTERACTIVA ---
# Esto generará la gráfica de pesos y el texto resaltado en tu Colab
exp_lsllama.show_in_notebook(text=True)

Experimento 4: LIME para BART
Este experimento mide la sensibilidad léxica. Si borramos una palabra técnica, ¿qué tanto deja BART de parecerse a su propia salida simplificada?

In [ ]:
import numpy as np
import pandas as pd
from lime.lime_text import LimeTextExplainer
from sentence_transformers import SentenceTransformer, util

# --- ESTO ASUME QUE YA TIENES LOS MODELOS CARGADOS DE ANTES ---
# nlp = spacy.load("es_core_news_sm")
#porque uso este model_sim
# model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# frase_orig = df_test.iloc[0, 0] # La frase médica larga
# resp_bart = df_test.iloc[0, 2] # La respuesta simplificada por BART

# --- CONFIGURACIÓN DE LIME ---
# class_names define las etiquetas en la gráfica de probabilidades
explainer_lime = LimeTextExplainer(class_names=['Diferente', 'Similar'])

# Función de predicción: LIME la usa para crear las perturbaciones
# porque usamos estas funciones
def predict_fn_bart(texts):
    # Comparamos semánticamente contra la salida de BART
    emb_target = model_sim.encode([resp_bart], convert_to_tensor=True)
    emb_inputs = model_sim.encode(texts, convert_to_tensor=True)
    scores = util.cos_sim(emb_inputs, emb_target).cpu().numpy().flatten()
    # Devolvemos formato de probabilidades [negativa, positiva]
    return np.array([[1-s, s] for s in scores])

# --- EJECUCIÓN DEL EXPERIMENTO ---
print("📊 Generando Auditoría LIME para BART (Experimento 4)...")
# exp_instance crea la explicación local. num_features=10 son las palabras clave.
exp_bart = explainer_lime.explain_instance(
    frase_orig,
    predict_fn_bart,
    num_features=10,
    num_samples=500 # Cantidad de perturbaciones para mayor precisión
)

# --- VISUALIZACIÓN INTERACTIVA  ---
# text=True asegura que se muestre el bloque de "Text with highlighted words"
# Al ejecutar esto, verás las tres columnas: Probabilidades, Gráfico con Pesos Numéricos y el Texto.
exp_bart.show_in_notebook(text=True)

Experimentos 5 y 6: SyntaxSHAP (Sintaxis Estricta)

In [ ]:
import pandas as pd

# Suponiendo que ya corriste los explicadores SHAP (shap_l y shap_b)
doc = nlp(frase_orig)
syntax_audit = []

for i, token in enumerate(doc):
    # Extraemos pesos de SHAP (Experimentos 1 y 2)
    w_llama = abs(shap_l.values[0][i]) if i < len(shap_l.values[0]) else 0
    w_bart = abs(shap_b.values[0][i]) if i < len(shap_b.values[0]) else 0
#porque tomamos esto?
    syntax_audit.append({
        "Token": token.text,
        "Función (Dep)": token.dep_,      # nsubj, root, obj...
        "Categoría (POS)": token.pos_,    # NOUN, VERB...
        "Padre (Head)": token.head.text,  # De quién depende
        "Peso_LSLlama": w_llama,
        "Peso_BART": w_bart
    })

df_syntax_tfm = pd.DataFrame(syntax_audit)

# Mostramos el resumen por Función Sintáctica
print("\n🔍 Resultados SyntaxSHAP: ¿Qué funciones gramaticales son más complejas?")
resumen_sintactico = df_syntax_tfm.groupby("Función (Dep)")[["Peso_LSLlama", "Peso_BART"]].mean()
display(resumen_sintactico.sort_values(by="Peso_LSLlama", ascending=False))

Traduccion y tabla maestra

5. Análisis de Resultados para responder la Pregunta de Investigación
Para responder "¿Por qué los modelos consideran que una palabra es compleja?", el código debe generar una tabla comparativa que analice:
Hallazgo en el Código
Interpretación Lingüística
Método de Soporte
Peso alto en LIME
La palabra es compleja por baja frecuencia en el corpus (criterio léxico puro).
LIME (Agnóstico)
Clusters en SHAP
La palabra es compleja por su entorno técnico médico (criterio proximal).
SHAP (Partition)
Peso alto en SyntaxShap + ADD
La palabra es compleja porque sobrecarga la estructura de la oración (criterio sintáctico)
.
SyntaxShap-W

Marco Teórico de la Decisión
Para responder por qué una palabra es compleja, dividimos el análisis en tres dimensiones:

Dimensión Léxica: La palabra es difícil por sí misma (ej. "antagonista").

Dimensión Contextual: La palabra es difícil porque está rodeada de tecnicismos (ej. "necrosis tumoral").

Dimensión Sintáctica: La palabra es difícil porque su función gramatical alarga la frase (ej. participios pasivos como "documentada").

5.1 Código para generar la Tabla de Hallazgos
Este bloque procesa los datos que ya se obtuvieron y los organiza para responder la pregunta de investigación:

In [ ]:
# --- GENERACIÓN DE LA TABLA DE HALLAZGOS PARA EL TFM ---

resultados_tfm = []

# Unimos la información de SyntaxSHAP con los pesos de LIME que calculamos
for i, row in df_syntax.iterrows():
    token = row['Token']
    pos = row['POS']
    p_llama = row['Peso_LSLlama']

    # Clasificación del motivo de complejidad según los experimentos
    if p_llama > 0.05:
        motivo = "Léxico-Semántico (Término crítico)"
        explicacion = "El modelo identifica el término como núcleo de la información técnica."
    elif pos in ['NOUN', 'ADJ'] and p_llama > 0.02:
        motivo = "Contextual (Proximal)"
        explicacion = "Complejidad derivada de la densidad de nombres técnicos."
    elif pos in ['VERB', 'AUX'] and p_llama > 0.01:
        motivo = "Estructural (Sintáctico)"
        explicacion = "Carga cognitiva por la estructura de la acción (Voz pasiva/tiempos comp)."
    else:
        motivo = "Baja relevancia"
        explicacion = "Palabra funcional o de relleno."

    resultados_tfm.append({
        "Token": token,
        "Categoría (POS)": pos,
        "Peso (SHAP)": f"{p_llama:.4f}",
        "Motivo de Complejidad": motivo,
        "Interpretación Lingüística": explicacion
    })

df_analisis_final = pd.DataFrame(resultados_tfm)

# Mostramos solo los términos que la IA consideró complejos (Peso > 0.01)
display(df_analisis_final[df_analisis_final['Peso (SHAP)'].astype(float) > 0.01].sort_values(by="Peso (SHAP)", ascending=False))

In [ ]:
import pandas as pd
import numpy as np
import shap
import spacy
import os
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer

# 1. CARGA DE DATOS SE SEGURIDAD (Ruta confirmada en tus pruebas previas)
path_real = 'drive/MyDrive/UC3m_TFM/resultados/test_claramed_final.csv'
df_test = pd.read_csv(path_real, sep=None, engine='python', on_bad_lines='skip')

# Extraemos la primera muestra usando índices para evitar KeyErrors
frase_orig = str(df_test.iloc[0, 0])
resp_lsllama = str(df_test.iloc[0, 1])
resp_bart = str(df_test.iloc[0, 2])

# 2. INICIALIZACIÓN DE MODELOS
nlp = spacy.load("es_core_news_sm")
model_sim = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
tokenizer_shap = AutoTokenizer.from_pretrained("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# Función base de scoring semántico
def get_score_base(texts, target):
    emb_target = model_sim.encode([target], convert_to_tensor=True)
    emb_inputs = model_sim.encode(texts, convert_to_tensor=True)
    return util.cos_sim(emb_inputs, emb_target).cpu().numpy().flatten()

# --- OBJETIVO 2 (SyntaxSHAP Avanzado y ADD) ---
print("⚙️ Analizando Estructura Sintáctica y Distancia de Dependencia (ADD)...")
doc = nlp(frase_orig)
sintaxis_data = []

for token in doc:
    # Distancia de dependencia: posición del token menos la posición de su núcleo sintáctico (head)
    distance = 0 if token.head == token else abs(token.i - token.head.i)
    sintaxis_data.append({
        "Token": token.text,
        "POS": token.pos_,
        "Dep": token.dep_,
        "Head": token.head.text,
        "ADD_Individual": distance
    })

df_sintaxis = pd.DataFrame(sintaxis_data)
add_promedio = df_sintaxis["ADD_Individual"].mean()

# --- CÁLCULO DE SHAP PARA OBTENER LOS PESOS DE IMPORTANCIA ---
print("🔮 Calculando atribuciones de SHAP para LSLlama...")
masker = shap.maskers.Text(tokenizer_shap)
explainer_sh = shap.Explainer(lambda x: get_score_base(x, resp_lsllama), masker)
shap_values = explainer_sh([frase_orig])

# Asignamos los pesos de SHAP a cada palabra mapeada por spaCy
pesos_lista = []
for i, row in df_sintaxis.iterrows():
    peso_t = abs(shap_values.values[0][i]) if i < len(shap_values.values[0]) else 0.0
    pesos_lista.append(peso_t)
df_sintaxis["Peso_SHAP"] = pesos_lista

# --- OBJETIVO 3 (Evaluación de Fidelidad div@10) ---
print("\n🧪 Ejecutando Evaluación de Fidelidad Semántica (Divergencia)...")

def calcular_divergencia_fidelidad(texto_original, dataframe_pesos, target_output, k=10):
    # Identificamos los K tokens considerados más complejos (mayor peso de SHAP)
    top_k_complejos = dataframe_pesos.sort_values(by="Peso_SHAP", ascending=False).head(k)["Token"].tolist()

    # Creamos un texto "perturbado" eliminando estos elementos
    palabras_originales = texto_original.split()
    palabras_filtradas = [w for w in palabras_originales if w not in top_k_complejos]
    texto_perturbado = " ".join(palabras_filtradas)

    # Medimos la similitud semántica antes y después de la perturbación contra la salida real
    score_original = get_score_base([texto_original], target_output)[0]
    score_perturbado = get_score_base([texto_perturbado], target_output)[0]

    # La fidelidad (div@K) mide cuánto divergen las salidas tras quitar el núcleo complejo
    divergencia = score_original - score_perturbado
    return score_original, score_perturbado, divergencia, top_k_complejos

sim_original, sim_perturbada, div_10, palabras_quitadas = calcular_divergencia_fidelidad(
    frase_orig, df_sintaxis, resp_lsllama, k=10
)

# --- INFORME METODOLÓGICO PARA EL TFM ---
print("\n" + "="*50)
print("📌 RESULTADOS DE AUDITORÍA CIENTÍFICA (XAI)")
print("="*50)
print(f"🔹 Distancia de Dependencia Promedio (ADD): {add_promedio:.2f}")
print(f"🔹 Similitud Original de la Respuesta:     {sim_original:.4f}")
print(f"🔹 Similitud sin Palabras Complejas:       {sim_perturbada:.4f}")
print(f"🔸 Fidelidad de la Explicación (div@10):   {div_10:.4f}")
print(f"🔹 Palabras Críticas Removidas (Top-10):    {palabras_quitadas}")
print("="*50)

# Gráfico de correlación entre Complejidad Estructural (ADD) y Atribución (SHAP)
plt.figure(figsize=(8, 5))
plt.scatter(df_sintaxis["ADD_Individual"], df_sintaxis["Peso_SHAP"], color='purple', alpha=0.7, edgecolors='black')
for i, txt in enumerate(df_sintaxis["Token"]):
    if df_sintaxis["Peso_SHAP"].iloc[i] > 0.01 or df_sintaxis["ADD_Individual"].iloc[i] > 4:
        plt.annotate(txt, (df_sintaxis["ADD_Individual"].iloc[i]+0.1, df_sintaxis["Peso_SHAP"].iloc[i]))

plt.title('Correlación: Carga Sintáctica (ADD) vs. Atribución XAI (SHAP)')
plt.xlabel('Distancia de Dependencia Sintáctica (ADD)')
plt.ylabel('Atribución de Complejidad (Peso SHAP)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

**Gráfico de Comparación Directa de Huellas Sintácticas.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Diccionario de traducción (Asegúrate de tenerlo definido)
traduccion_dep = {
    'nsubj': 'Sujeto',
    'root': 'Verbo Principal',
    'obj': 'Objeto Directo',
    'iobj': 'Objeto Indirecto',
    'amod': 'Adjetivo (Modificador)',
    'advmod': 'Adverbio',
    'appos': 'Aposición (Técnica)',
    'conj': 'Conjunción',
    'det': 'Determinante',
    'prep': 'Preposición',
    'pobj': 'Objeto de Preposición',
    'flat': 'Nombre Compuesto'
}

# 2. Preparar los datos usando los nombres de tu PARTE 8
# Agrupamos por la columna "Función (Dep)" que es la que creaste
df_grouped = df_syntax_tfm.groupby("Función (Dep)")[["Peso_LSLlama", "Peso_BART"]].mean().reset_index()

# Mapeamos la traducción (si no existe la etiqueta en el dict, deja el nombre original)
df_grouped['Descripcion'] = df_grouped['Función (Dep)'].map(traduccion_dep).fillna(df_grouped['Función (Dep)'])

# Ordenar por importancia para LSLlama
df_grouped = df_grouped.sort_values(by="Peso_LSLlama", ascending=True)

# 3. Configuración del gráfico
labels = df_grouped['Descripcion']
y = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 8))

rects1 = ax.barh(y - width/2, df_grouped['Peso_LSLlama'], width, label='LSLlama', color='#3498db', alpha=0.8)
rects2 = ax.barh(y + width/2, df_grouped['Peso_BART'], width, label='BART', color='#e74c3c', alpha=0.8)

# Estética UC3M
ax.set_xlabel('Peso de Atribución (Importancia en la Simplificación)', fontsize=12)
ax.set_title('Huella Sintáctica: LSLlama vs BART\n¿Qué estructuras consideran más complejas?', fontsize=15, pad=20)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=10)
ax.legend()
ax.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("comparativa_syntaxshap_modelos.png", dpi=300)
plt.show()

# Análisis de hallazgos
print("\n🔍 HALLAZGO PARA TU TFM:")
max_llama = df_grouped.loc[df_grouped['Peso_LSLlama'].idxmax(), 'Descripcion']
max_bart = df_grouped.loc[df_grouped['Peso_BART'].idxmax(), 'Descripcion']
print(f"- LSLlama identifica mayor complejidad en: {max_llama}")
print(f"- BART identifica mayor complejidad en: {max_bart}")

Experimento de Consolidación (LIME vs SHAP vs SyntaxSHAP)

In [ ]:
import shap

# 1. Configuramos el explainer para LSLlama
# Usamos el modelo de similitud como la función que SHAP debe explicar
explainer_shap_llama = shap.Explainer(predict_fn_lsllama, masker=shap.maskers.Text(tokenizer=llama_tokenizer))

# 2. Configuramos el explainer para BART
explainer_shap_bart = shap.Explainer(predict_fn_bart, masker=shap.maskers.Text(tokenizer=bart_tokenizer))

print("✅ SHAP Explainers definidos para LSLlama y BART.")
# 1. Definimos la frase de estudio globalmente (primera frase del dataset)
frase_estudio = df_final.iloc[0]["Original (ClaraMeD)"]
res_llama = df_final.iloc[0]["LSLlama (IA)"]
res_bart = df_final.iloc[0]["BART (IA)"]

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# --- NUEVA FUNCIÓN AÑADIDA PARA EXTRAER PESOS SHAP ESCALARES ---
def extraer_pesos_escalares(shap_obj):
    # Asume que shap_obj es un objeto shap.Explanation (ej. de shap.Text explainer)
    # Los valores son típicamente un array 2D, donde el primer elemento es la instancia
    return shap_obj.values[0]

# --- 1. FUNCIÓN DE LIMPIEZA DE PESOS LIME ---
def extraer_pesos_lime(frase, predict_fn):
    exp = explainer_lime.explain_instance(frase, predict_fn, num_features=10, num_samples=500)
    lista_pesos = exp.as_list()
    return pd.DataFrame(lista_pesos, columns=['Palabra', 'Peso']).sort_values('Peso')

# --- 2. CONFIGURACIÓN DEL GRAN PANEL (3x2) ---
fig, axes = plt.subplots(2, 3, figsize=(24, 14))

# --- FILA 1: LSLLAMA ---
print("📊 Dibujando LSLlama...")
# 1. LIME
df_lime_l = extraer_pesos_lime(frase_estudio, predict_fn_lsllama)
df_lime_l.tail(8).plot(kind='barh', x='Palabra', y='Peso', ax=axes[0,0], color='teal')
axes[0,0].set_title("1. LSLlama: LIME (Sensibilidad Local)")

# 2. SHAP
tokens_spacy_l = [t.text for t in nlp(frase_estudio)]
pesos_l = extraer_pesos_escalares(shap_l)
min_l = min(len(tokens_spacy_l), len(pesos_l))
df_shap_l = pd.DataFrame({'Token': tokens_spacy_l[:min_l], 'S': pesos_l[:min_l]}).sort_values('S')
df_shap_l.tail(8).plot(kind='barh', x='Token', y='S', ax=axes[0,1], color='skyblue')
axes[0,1].set_title("2. LSLlama: SHAP (Atribución Justa)")

# 3. SyntaxSHAP (USANDO TU VARIABLE resumen_sintactico)
# Aquí usamos la columna de LSLlama que creaste en la Parte 8
resumen_sintactico[['Peso_LSLlama']].sort_values(by="Peso_LSLlama").tail(8).plot(kind='barh', ax=axes[0,2], color='blue')
axes[0,2].set_title("3. LSLlama: SyntaxSHAP (Sintaxis)")

# --- FILA 2: BART ---
print("📊 Dibujando BART...")
# 1. LIME para BART
df_lime_b = extraer_pesos_lime(frase_estudio, predict_fn_bart)
df_lime_b.tail(8).plot(kind='barh', x='Palabra', y='Peso', ax=axes[1,0], color='darkorange')
axes[1,0].set_title("1. BART: LIME (Sensibilidad Local)")

# 2. SHAP para BART
tokens_spacy_b = [t.text for t in nlp(frase_estudio)] # Added this line
pesos_b = extraer_pesos_escalares(shap_b)
min_b = min(len(tokens_spacy_b), len(pesos_b))
df_shap_b = pd.DataFrame({'Token': tokens_spacy_b[:min_b], 'S': pesos_b[:min_b]}).sort_values('S')
df_shap_b.tail(8).plot(kind='barh', x='Token', y='S', ax=axes[1,1], color='lightcoral')
axes[1,1].set_title("2. BART: SHAP (Atribución Justa)")

# 3. SyntaxSHAP para BART (USANDO TU VARIABLE resumen_sintactico)
# Seleccionamos solo la columna de BART que generamos en la Parte 8
resumen_sintactico[['Peso_BART']].sort_values(by="Peso_BART").tail(8).plot(kind='barh', ax=axes[1,2], color='red')
axes[1,2].set_title("3. BART: SyntaxSHAP (Sintaxis)")
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.suptitle(f"Auditoría de Explicabilidad Médica: {frase_estudio[:70]}...", fontsize=18)
plt.show()

Celda de Consolidación: Cruce Analítico Triple XAI vs EASSE

In [ ]:
import numpy as np
import pandas as pd
import spacy
from easse.utils import get_word_level_edit_operations

# 1. Cargar el motor sintáctico en español para el análisis de SyntaxSHAP
nlp = spacy.load("es_core_news_sm")

# Lista global para consolidar el cruce estadístico sin asunciones
matriz_analitica_xai = []

# Iteración síncrona sobre cada una de las frases del experimento
for idx, row in enumerate(resultados_tfm):
    src_text = row['Original']

    # NOTA DE SELECCIÓN: Cambiar por row['BART (IA)'] según el modelo que estés auditando en este bucle
    pred_text = row['LSLlama (IA)']

    # -------------------------------------------------------------------------
    # CAPA EXTRÍNSECA (MÉTRICA): Extracción de operaciones físicas de EASSE
    # -------------------------------------------------------------------------
    easse_ops = get_word_level_edit_operations(src_text, pred_text)

    # Extraer tokens del texto original involucrados en transformaciones específicas
    palabras_replace = [str(op[1]).lower().strip(".,;:()") for op in easse_ops if op[0] == 'REPLACE']
    palabras_delete  = [str(op[1]).lower().strip(".,;:()") for op in easse_ops if op[0] == 'DELETE']
    palabras_move    = [str(op[1]).lower().strip(".,;:()") for op in easse_ops if op[0] == 'MOVE']

    # -------------------------------------------------------------------------
    # CAPA INTRÍNSECA (XAI 1): Extracción de Pesos Locales de LIME
    # -------------------------------------------------------------------------
    # 'exp' es tu objeto LIME local generado para la frase actual en el notebook
    # Extrae una lista de tuplas: (token, peso)
    dict_lime_local = {str(tok).lower().strip(".,;:()"): p for tok, p in exp.as_list()}

    # -------------------------------------------------------------------------
    # CAPA INTRÍNSECA (XAI 2): Extracción de Valores Aditivos de SHAP
    # -------------------------------------------------------------------------
    # 'shap_values_instance' representa el slice de tu shap.Explanation para la frase actual (ej. shap_values_llama[idx])
    # Extraemos la correspondencia exacta entre datos crudos y sus valores de Shapley
    dict_shap_local = {}
    for tok_shap, val_shap in zip(shap_values_instance.data, shap_values_instance.values):
        tok_key = str(tok_shap).lower().strip(".,;:()")
        dict_shap_local[tok_key] = val_shap

    # -------------------------------------------------------------------------
    # CAPA INTRÍNSECA (XAI 3 + SINTAXIS): Construcción de SyntaxSHAP vía spaCy
    # -------------------------------------------------------------------------
    doc_spacy = nlp(src_text)

    for token in doc_spacy:
        token_str = token.text
        token_key = token_str.lower().strip(".,;:()")

        # Mapear los pesos correspondientes de LIME y SHAP si existen para este token
        peso_lime = dict_lime_local.get(token_key, 0.0)
        peso_shap = dict_shap_local.get(token_key, 0.0)

        # Extracción de etiquetas sintácticas estructurales para el análisis de SyntaxSHAP
        rol_dependencia = token.dep_  # Etiqueta sintáctica (ej: nsubj, obj, amod)
        categoria_pos   = token.pos_  # Categoría gramatical (ej: NOUN, VERB, ADJ)

        # -------------------------------------------------------------------------
        # ALINEACIÓN Y DIAGNÓSTICO DE CORRELACIÓN
        # -------------------------------------------------------------------------
        if token_key in palabras_replace:
            operacion_detectada = "REPLACE"
        elif token_key in palabras_delete:
            operacion_detectada = "DELETE"
        elif token_key in palabras_move:
            operacion_detectada = "MOVE"
        else:
            operacion_detectada = "COPY" # Preservada o mantenida idéntica

        matriz_analitica_xai.append({
            "id_instancia": idx,
            "token_original": token_str,
            "categoria_pos": categoria_pos,
            "dependencia_syntaxshap": rol_dependencia,
            "operacion_EASSE": operacion_detectada,
            "atribucion_LIME": peso_lime,
            "valor_SHAP": peso_shap,
            "magnitud_SyntaxSHAP": abs(peso_shap) # Sintaxis analiza la magnitud del impacto neto
        })

# Consolidar todas las dimensiones interpretativas en un DataFrame único
df_diagnostico_total = pd.DataFrame(matriz_analitica_xai)
print("Matriz unificada completada con éxito. Dimensiones del DataFrame:", df_diagnostico_total.shape)
df_diagnostico_total.head(15)

Celda de Visualización y Métricas Estadísticas para el TFM

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Análisis estadístico descriptivo: Comparar promedios de impacto absoluto de LIME y SHAP por cada operación de EASSE
df_diagnostico_total['abs_LIME'] = df_diagnostico_total['atribucion_LIME'].abs()
df_diagnostico_total['abs_SHAP'] = df_diagnostico_total['valor_SHAP'].abs()

print("\n📊 IMPACTO PROMEDIO DE LOS COMPONENTES XAI SEGÚN LA OPERACIÓN FÍSICA:")
print(df_diagnostico_total.groupby('operacion_EASSE')[['abs_LIME', 'abs_SHAP']].mean())

print("\n🌿 CRUCE SYNTAXSHAP: Top 10 de dependencias sintácticas con mayor peso en operaciones de sustitución o borrado:")
reporte_sintactico = df_diagnostico_total.groupby(['dependencia_syntaxshap', 'operacion_EASSE'])['magnitud_SyntaxSHAP'].mean().unstack().fillna(0)
print(reporte_sintactico.sort_values(by='REPLACE', ascending=False).head(10))

# 2. Plotting del Gráfico Boxplot Doble para el marco metodológico
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(data=df_diagnostico_total, x='operacion_EASSE', y='atribucion_LIME', ax=axes[0], palette='Set2')
axes[0].axhline(0, color='grey', linestyle='--', alpha=0.7)
axes[0].set_title('Atribución Local de Características (LIME)')
axes[0].set_xlabel('Operación Registrada por EASSE')
axes[0].set_ylabel('Peso LIME')

sns.boxplot(data=df_diagnostico_total, x='operacion_EASSE', y='valor_SHAP', ax=axes[1], palette='Pastel1')
axes[1].axhline(0, color='grey', linestyle='--', alpha=0.7)
axes[1].set_title('Valores de Atribución Aditiva de Shapley (SHAP)')
axes[1].set_xlabel('Operación Registrada por EASSE')
axes[1].set_ylabel('Valor de Shapley')

plt.suptitle('Cruce Metodológico de Explicabilidad (XAI) y Operaciones de Edición (EASSE)', fontsize=14)
plt.tight_layout()
plt.show()

**Conclusiones**

SHAP te da un número ($\phi$) para cada palabra."No es que el modelo LSLlama se confunda con la palabra 'alfa' por ser difícil, sino que se confunde porque esa palabra forma parte de una Aposición (appos). Esto demuestra que el modelo tiene dificultades procesando aclaraciones gramaticales complejas, no solo vocabulario técnico."

"A través de los valores SHAP se observa que LSLlama no solo se detiene en términos complejos aislados como 'antagonistas', sino que asigna una carga significativa a conectores como 'pero' y 'actual'. Esto demuestra que el modelo está procesando la lógica condicional del texto médico (si pasó X o si pasa Y), lo cual es fundamental para una simplificación correcta que no altere el sentido del tratamiento."

**Preguntas **

Al usar SyntaxSHAP he visto que las preposiciones y conectores tienen mucha importancia. ¿Cree que deberíamos interpretar esto como una señal de que el modelo (LSLlama/BART) está priorizando la preservación de la estructura lógica sobre la sustitución de términos complejos?"